# 03 - Differential Expression Analysis

This notebook compares **Cancer** with **Healthy** samples using pyDESeq2. It loads the filtered raw integer count matrix saved by `02_data_qc.ipynb` as `data/processed/raw_merged_filtered.csv`.

Differential expression can rank or select genes for dimensionality reduction. For unbiased ML evaluation, however, do **not** use genes selected by a DEA fitted on the full dataset before splitting: labels from the eventual test set would influence feature selection. Fit DEA separately using each training split (inside cross-validation) and apply that split's selected genes to its held-out samples.


In [ ]:
import sys
sys.path.append('..')  # repository root when running from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats


## 1. Load filtered raw counts and metadata

The filtered count matrix should have samples in rows and genes in columns, with the same sample identifiers as the metadata. pyDESeq2 requires raw integer counts; do not use the VST matrix as model input.

In [ ]:
data_dir = '../data/processed'

counts_filtered = pd.read_csv(
    f'{data_dir}/raw_merged_filtered.csv',
    index_col=0,
)
metadata = pd.read_csv(
    f'{data_dir}/metadata_merged.csv',
    index_col=0,
)

if not counts_filtered.index.is_unique or not metadata.index.is_unique:
    raise ValueError('Sample identifiers must be unique in counts and metadata.')
if not counts_filtered.index.isin(metadata.index).all():
    raise ValueError('Some count-matrix samples are missing from metadata.')
if 'Condition' not in metadata.columns:
    raise ValueError('Metadata must contain a Condition column.')

# Keep samples in count-matrix order and align metadata exactly.
metadata = metadata.loc[counts_filtered.index].copy()
assert metadata.index.equals(counts_filtered.index)

print(f'Filtered count matrix shape (samples, genes): {counts_filtered.shape}')
display(metadata['Condition'].value_counts())


## 2. Prepare counts and condition metadata

Only the two requested conditions are included. The contrast is **Cancer relative to Healthy**, so a positive log2 fold change means higher expression in Cancer.

In [ ]:
conditions = ['Healthy', 'Cancer']
if metadata['Condition'].isna().any():
    raise ValueError('Condition contains missing values.')

keep_samples = metadata['Condition'].isin(conditions)
counts = counts_filtered.loc[keep_samples].copy()
meta = metadata.loc[keep_samples, ['Condition']].copy()
meta['Condition'] = pd.Categorical(meta['Condition'], categories=conditions)

if set(meta['Condition'].dropna().astype(str)) != set(conditions):
    raise ValueError('Both Healthy and Cancer samples are required for the comparison.')
if not np.isfinite(counts.to_numpy(dtype=float)).all() or (counts.to_numpy(dtype=float) < 0).any():
    raise ValueError('Raw counts must be finite and non-negative.')
if not np.equal(counts.to_numpy(dtype=float), np.floor(counts.to_numpy(dtype=float))).all():
    raise ValueError('Counts must be integers; do not use VST values as pyDESeq2 input.')
counts = counts.astype('int64')

print('Samples per condition:')
display(meta['Condition'].value_counts().reindex(conditions))
print(f'Count matrix supplied to pyDESeq2: {counts.shape}')


## 3. Fit the pyDESeq2 model

The design includes condition only because the available metadata contains no batch covariate. If known batch or other covariates are available, review the design before interpreting results.

In [ ]:
dds = DeseqDataSet(
    counts=counts,
    metadata=meta,
    design='~Condition',
    refit_cooks=True,
    n_cpus=4,
)
dds.deseq2()

# Cancer versus Healthy: positive log2FoldChange indicates higher expression in Cancer.
stat_res = DeseqStats(
    dds,
    contrast=['Condition', 'Cancer', 'Healthy'],
    n_cpus=4,
)
stat_res.summary()
results = stat_res.results_df.sort_values('padj', na_position='last').copy()
display(results.head())


## 4. Summarize, select, and save results

Results include the estimated log2 fold change, standard error, test statistic, p-value, and Benjamini-Hochberg adjusted p-value. The following section uses adjusted p-value < 0.05 as the gene selection rule. For ML performance estimates, repeat this feature-selection step using training data only within each split.

In [ ]:
alpha = 0.05
significant = results['padj'].notna() & (results['padj'] < alpha)
selected_genes = results.index[significant]

print(f'Genes tested: {len(results):,}')
print(f'Genes with adjusted p-value < {alpha}: {len(selected_genes):,}')
print('Top results by adjusted p-value:')
display(results.loc[significant].head(20))

results.to_csv(f'{data_dir}/differential_expression_cancer_vs_healthy.csv', index_label='gene')
pd.Series(selected_genes, name='gene').to_csv(
    f'{data_dir}/differential_expression_selected_genes_cancer_vs_healthy.csv',
    index=False,
)
print(f'Saved full results to {data_dir}/differential_expression_cancer_vs_healthy.csv')
print(f'Saved selected gene list to {data_dir}/differential_expression_selected_genes_cancer_vs_healthy.csv')


## 5. Volcano plot

The x-axis shows the Cancer versus Healthy log2 fold change. The y-axis shows the negative log10 adjusted p-value; genes without an adjusted p-value are omitted from the plot.

In [ ]:
plot_df = results.dropna(subset=['padj', 'log2FoldChange']).copy()
plot_df['neg_log10_padj'] = -np.log10(plot_df['padj'].clip(lower=np.finfo(float).tiny))
plot_df['Significance'] = np.where(plot_df['padj'] < alpha, f'adjusted p-value < {alpha}', 'Not significant')

fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(
    data=plot_df,
    x='log2FoldChange',
    y='neg_log10_padj',
    hue='Significance',
    hue_order=[f'adjusted p-value < {alpha}', 'Not significant'],
    palette={f'adjusted p-value < {alpha}': '#c0392b', 'Not significant': '#7f8c8d'},
    linewidth=0,
    s=18,
    alpha=0.75,
    ax=ax,
)
ax.axhline(-np.log10(alpha), color='black', linestyle='--', linewidth=1)
ax.axvline(0, color='black', linewidth=1)
ax.set(title='Differential expression: Cancer vs Healthy', xlabel='log2 fold change (Cancer / Healthy)', ylabel='-log10 adjusted p-value')
plt.tight_layout()
plt.show()


## Interpretation and ML notes

- Positive log2 fold changes indicate higher estimated expression in Cancer; negative values indicate higher expression in Healthy.
- Inference uses raw integer counts from `raw_merged_filtered.csv`, which already has the QC count threshold applied.
- The saved adjusted-p-value-selected gene list can support exploratory dimensionality reduction. Do not use this full-cohort list to report held-out ML performance: perform DEA and feature selection within each training split/fold, then transform the corresponding validation/test data using only those selected genes.
- The model uses condition alone. Interpret results with the available sample metadata and study design in mind.
